In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import torch
print(f"PyTorch : {torch.__version__}")
print(f"Device     : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")


In [ ]:
!pip install -q huggingface_hub

from google.colab import userdata
from huggingface_hub import login

# Lấy token từ Secrets của Colab
hf_token = userdata.get('tuankun08')

# Tiến hành đăng nhập
login(token=hf_token)
print("Đã kết nối thành công với Hugging Face!")


In [ ]:
from huggingface_hub import snapshot_download

repo_id = "tuankun08/Dataset"

# Tải toàn bộ repository/folder về thư mục '/content/Dataset' trên Colab
snapshot_download(
    repo_id=repo_id,
    allow_patterns="Dataset/*",
    repo_type="dataset",
    local_dir="/content/Dataset"
)


In [ ]:
!git clone https://github.com/tuananhh08/Hallsensor_final.git


In [ ]:
!pip install -q scikit-learn pandas numpy matplotlib

DATA_DIR = "/content/Dataset/Dataset"
REPO     = "/content/Hallsensor_final/Benchmark/MobilePosenet"
RUNS     = "/content/drive/MyDrive/MobilePoseNet_runs"
CACHE    = "/content/mobileposenet_cache"

# Finetune data
RAW_VOLTAGE           = f"{DATA_DIR}/Grid_add_Random_data.csv"
CLEAN_VOLTAGE         = f"{DATA_DIR}/Grid_add_Random_data_computed.csv"

# Grid data
GRID_VOLTAGE          = f"{DATA_DIR}/Grid_data.csv"
GRID_CLEAN_VOLTAGE    = f"{DATA_DIR}/Grid_data_computed.csv"

# Helix data
HELIX_VOLTAGE         = f"{DATA_DIR}/Helix_data_2.csv"
HELIX_CLEAN_VOLTAGE   = f"{DATA_DIR}/Helix_data_2_computed.csv"

# Random data
RANDOM_VOLTAGE        = f"{DATA_DIR}/Random_data.csv"
RANDOM_CLEAN_VOLTAGE  = f"{DATA_DIR}/Random_data_computed.csv"

# Label
RAW_LABEL             = f"{DATA_DIR}/Grid_add_Random_points_coordinates.csv"
GRID_LABEL            = f"{DATA_DIR}/Grid_points_coordinates.csv"
HELIX_LABEL           = f"{DATA_DIR}/Helix_points_coordinates_2.csv"
RANDOM_LABEL          = f"{DATA_DIR}/Random_points_coordinates.csv"

# Synthetic data ~700k data
SYNTHETIC_VOLTAGE = f"{DATA_DIR}/synthetic_grid_data.csv"
SYNTHETIC_LABEL   = f"{DATA_DIR}/synthetic_grid_coordinates.csv"



In [ ]:
# TRAIN MOBILEPOSENET ON THEORETICAL DATA
!python $REPO/run.py \
  --synthetic-voltage "$SYNTHETIC_VOLTAGE" \
  --synthetic-label "$SYNTHETIC_LABEL" \
  --raw-voltage "$RAW_VOLTAGE" \
  --clean-voltage "$CLEAN_VOLTAGE" \
  --raw-label "$RAW_LABEL" \
  --ckpt-dir "$RUNS/MobilePoseNet_training" \
  --cache-dir "$CACHE" \
  --epochs 400 \
  --batch-size 512 \
  --lr 0.00048438278642785697 \
  --beta 0.2234016585319006 \
  --val-ratio 0.02 \
  --device cuda \
  --no-compile

In [ ]:
# TEST INFERENCE ON RAW HELIX MEASUREMENTS
import os
os.makedirs(f"{RUNS}/evaluation", exist_ok=True)

!python $REPO/evaluation/evaluate.py \
  --test-voltage "$HELIX_VOLTAGE" \
  --test-label "$HELIX_LABEL" \
  --ckpt-dir "$RUNS/training" \
  --checkpoint "mobileposenet_best.pt" \
  --scalers "$RUNS/training/scalers.pkl" \
  --batch-size 512 \
  --device cuda \
  --out "$RUNS/evaluation/mobileposenet_trajectory.png" \
  --out-csv "$RUNS/evaluation/mobileposenet_predictions.csv"
